In [1]:
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras import models

2026-05-12 12:59:23.957229: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778590764.347863      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778590764.452698      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778590765.423157      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778590765.423192      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778590765.423195      57 computation_placer.cc:177] computation placer alr

In [2]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient


user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HUGGINGFACE_KEY")

login(token=hf_token)

In [3]:
class pc_block_d(layers.Layer):
    def __init__(self, filters, kernel_size=4, strides=2):
        super(pc_block_d, self).__init__()
        self.filters = filters
        self.kernel_size = kernel_size
        self.strides = strides
        
        # Forward pass downsampling
        self.conv = layers.Conv2D(filters, kernel_size, strides=strides, padding='same')
        self.ln = layers.LayerNormalization()
        self.leaky = layers.LeakyReLU(0.2)
        
        # Upsampling geometry (no trainable weights, safe in __init__)
        self.upsample = layers.UpSampling2D(size=(strides, strides), interpolation='nearest')
        
        # [IMPROVEMENT]: Delayed instantiation. We will define this in build()
        self.up_conv = None

    def build(self, input_shape):
        # [IMPROVEMENT]: Dynamically extract the exact incoming channel count
        in_channels = input_shape[-1]
        
        # Create the reconstruction convolution to perfectly match the input depth
        self.up_conv = layers.Conv2D(in_channels, kernel_size=3, padding='same')
        super(pc_block_d, self).build(input_shape)

    def call(self, x):
        # 1. Forward feature extraction
        features = self.leaky(self.ln(self.conv(x)))
        
        # 2. Local reconstruction (checkerboard-proof)
        reconstructed_grid = self.upsample(features)
        reconstruction = self.up_conv(reconstructed_grid)
        
        # 3. Predictive error calculation
        error = x - reconstruction
        
        return features, reconstruction, error

In [4]:
class pc_discriminator(models.Model):
    def __init__(self):
        super(pc_discriminator, self).__init__()
        self.block1 = pc_block_d(64)
        self.block2 = pc_block_d(128)
        self.block3 = pc_block_d(256)
        self.block4 = pc_block_d(512)

        self.flatten = layers.Flatten()
        self.final_dense = layers.Dense(1)

    def call(self, x):
        # Unpack the 3 variables: features (h), reconstruction (rec), and error (e)
        h1, rec1, e1 = self.block1(x)
        h2, rec2, e2 = self.block2(h1)
        h3, rec3, e3 = self.block3(h2)
        h4, rec4, e4 = self.block4(h3)

        # Calculate total error (can still be useful for logging convergence)
        total_error = tf.reduce_mean(tf.square(e1)) + \
                      tf.reduce_mean(tf.square(e2)) + \
                      tf.reduce_mean(tf.square(e3)) + \
                      tf.reduce_mean(tf.square(e4))
        
        # Final adversarial score
        score = self.final_dense(self.flatten(h4))
        
        # Collect local states for manual weight updates outside the graph
        features = [h1, h2, h3, h4]
        reconstructions = [rec1, rec2, rec3, rec4]
        errors = [e1, e2, e3, e4]
        
        return score, total_error, features, reconstructions, errors

In [5]:
import tensorflow as tf
from tensorflow.keras import layers

class pc_block_g(layers.Layer):
    def __init__(self, filters, kernel_size=4, strides=2):
        super(pc_block_g, self).__init__()
        self.filters = filters
        self.kernel_size = kernel_size
        self.strides = strides
        
        # --- THE ARCHITECTURE UPGRADE ---
        # 1. Replaced Conv2DTranspose with pure geometric upsampling
        self.upsample = layers.UpSampling2D(size=(strides, strides), interpolation='nearest')
        
        # 2. Added standard Conv2D for smooth feature learning
        self.conv_up = layers.Conv2D(filters, kernel_size, padding='same', use_bias=False)
        
        self.ln = layers.LayerNormalization()
        self.relu = layers.ReLU()
        
        self.predict_back_v2 = None 

    def build(self, input_shape):
        in_channels = input_shape[-1]
        self.predict_back_v2 = layers.Conv2D(in_channels, self.kernel_size, 
                                             strides=self.strides, padding='same')
        super(pc_block_g, self).build(input_shape)

    def call(self, x, training=True):
        # 1. Scale up the grid safely without overlapping gradient issues
        upscaled = self.upsample(x)
        
        # 2. Learn and refine the features smoothly
        features = self.conv_up(upscaled)
        
        # 3. Apply normalization and activation
        h = self.relu(self.ln(features))
        
        # 4. The internal reflection pathway (remains unchanged)
        prediction = self.predict_back_v2(h)
        error = x - prediction
        
        return h, prediction, error

In [6]:
import tensorflow as tf
from tensorflow.keras import models, layers

class pc_generator(models.Model):
    def __init__(self):
        super(pc_generator, self).__init__()
        # Defining the hierarchy of PC blocks 
        self.block1 = pc_block_g(512, strides=4) 
        self.block2 = pc_block_g(256)
        self.block3 = pc_block_g(128)
        self.block4 = pc_block_g(64)

        # [CHANGED]: Replaced Conv2DTranspose with Sequential UpSampling + Conv2D
        self.final_layer = tf.keras.Sequential([
            layers.UpSampling2D(size=(2, 2), interpolation='nearest'),
            layers.Conv2D(
                3, 4, padding='same', 
                kernel_initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.02),
                use_bias=False, activation='tanh'
            )
        ])

    def call(self, x, training=True):
        h1, pred1, e1 = self.block1(x, training=training)
        h2, pred2, e2 = self.block2(h1, training=training)
        h3, pred3, e3 = self.block3(h2, training=training)
        h4, pred4, e4 = self.block4(h3, training=training)
        
        # This now safely upsamples and convolves without checkerboarding
        output = self.final_layer(h4)
        
        total_gen_error = tf.reduce_mean(tf.square(e1)) + \
                          tf.reduce_mean(tf.square(e2)) + \
                          tf.reduce_mean(tf.square(e3)) + \
                          tf.reduce_mean(tf.square(e4))
        
        features = [h1, h2, h3, h4]
        predictions = [pred1, pred2, pred3, pred4]
        errors = [e1, e2, e3, e4]
        
        return output, total_gen_error, features, predictions, errors

In [7]:
import tensorflow as tf
from huggingface_hub import snapshot_download

# 1. Download from HF
path = snapshot_download(repo_id="RinKana/PC-GAN-V3-A", allow_patterns="checkpoints/*")
latest_ckpt_path = tf.train.latest_checkpoint(f"{path}/checkpoints")

# 2. Re-build the "Body"
generator = pc_generator()
discriminator = pc_discriminator()

# Initialize the layers with dummy data to trigger the build() methods
noise_testparam = tf.random.normal([1, 1, 1, 100])
picture_testparam = tf.random.normal([1, 64, 64, 3])

_ = generator(noise_testparam, training=False)
_ = discriminator(picture_testparam)

print("Architecture built. Checking summaries...")
generator.summary()
discriminator.summary()

# --- PHASE 3 CHANGE: Optimizers Completely Removed ---
# We no longer instantiate tf.keras.optimizers.Adam here.

# 3. Create the Checkpoint Wrapper (Model Weights Only)
ckpt = tf.train.Checkpoint(generator=generator,
                           discriminator=discriminator)

# 4. Restore the state
# --- PHASE 3 CHANGE: Added .expect_partial() ---
# This injects the Phase 2 structural weights but safely ignores 
# the orphaned Adam memory and the mismatched predict_back layer.
status = ckpt.restore(latest_ckpt_path).expect_partial()

if status:
    print("Checkpoint restored successfully. Global backprop dropped, Amadeus is ready for Phase 3 local updates!")

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

I0000 00:00:1778590806.084874      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1778590806.090842      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1778590808.426397      57 cuda_dnn.cc:529] Loaded cuDNN version 91002


Architecture built. Checking summaries...


Model: "pc_generator"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ pc_block_g (pc_block_g)         │ ?                      │     1,639,524 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pc_block_g_1 (pc_block_g)       │ ?                      │     4,195,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pc_block_g_2 (pc_block_g)       │ ?                      │     1,049,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pc_block_g_3 (pc_block_g)       │ ?                      │       262,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (1, 64, 64, 3)         │         3,072 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ pc_block_d (pc_block_d)         │ ?                      │         4,995 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pc_block_d_1 (pc_block_d)       │ ?                      │       205,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pc_block_d_2 (pc_block_d)       │ ?                      │       820,096 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pc_block_d_3 (pc_block_d)       │ ?                      │     3,278,592 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (1, 8192)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (1, 1)                 │         8,193 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,317,124 (16.47 MB)

 Trainable params: 4,317,124 (16.47 MB)

 Non-trainable params: 0 (0.00 B)

Checkpoint restored successfully. Global backprop dropped, Amadeus is ready for Phase 3 local updates!


In [8]:
import tensorflow as tf

# 1. Define local path and parameters
# Note: Keras needs the path to the folder ABOVE the images folder 
# if your images are in /images/subfolder. If all images are in /images, 
# point to the parent of /images.
DATASET_PATH = "/kaggle/input/datasets/splcher/animefacedataset"
IMG_HEIGHT, IMG_WIDTH = 64, 64
BATCH_SIZE = 32

# 2. Load the dataset using Keras utilities
# label_mode=None ensures it only returns images, not labels
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    label_mode=None,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    shuffle=True
)

# 3. Apply Normalization [-1, 1] for your PC-GAN
# (x - 127.5) / 127.5 is standard for Tanh activation
train_ds = train_ds.map(lambda x: (x - 127.5) / 127.5)

# 4. Performance optimizations
# .cache() keeps images in RAM so they aren't re-read from disk every epoch
# .prefetch() overlaps data preprocessing with model execution
train_ds = train_ds.cache().prefetch(buffer_size=tf.data.AUTOTUNE)

print("Dataset successfully loaded for single-device training.")


Found 63565 files.
Dataset successfully loaded for single-device training.


In [9]:
for batch in train_ds.take(1):
    print("Batch shape:", batch.shape)
    print("Min value:", tf.reduce_min(batch).numpy())
    print("Max value:", tf.reduce_max(batch).numpy())

Batch shape: (32, 64, 64, 3)
Min value: -1.0
Max value: 1.0


In [10]:
import tensorflow as tf

# 1. SETUP CHECKPOINTS
checkpoint_dir = './training_checkpoints'
ckpt = tf.train.Checkpoint(generator=generator, discriminator=discriminator)
checkpoint_manager = tf.train.CheckpointManager(ckpt, checkpoint_dir, max_to_keep=3)
cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)

# 2. LOCALIZED ADAM-W HYPERPARAMETERS
# [CHANGED]: Increased LR from 2e-5 to 1e-4 to force mutation and break the "building" pattern
MANUAL_LR = tf.constant(1e-4, dtype=tf.float32) 
BETA_1 = tf.constant(0.9, dtype=tf.float32)
BETA_2 = tf.constant(0.999, dtype=tf.float32)
EPSILON = tf.constant(1e-7, dtype=tf.float32)
WEIGHT_DECAY = tf.constant(1e-4, dtype=tf.float32)

# --- THE FIX: NEW HYPERPARAMETERS TO PREVENT COLLAPSE ---
LAMBDA_PC = tf.constant(0.1, dtype=tf.float32)     # Scales down the internal U-Net error
CLIP_NORM = tf.constant(1.0, dtype=tf.float32)     # Hard ceiling for gradient shocks

# 3. INITIALIZE LOCAL MEMORY
generator_momentums = [tf.Variable(tf.zeros_like(var), trainable=False) for var in generator.trainable_variables]
generator_velocities = [tf.Variable(tf.zeros_like(var), trainable=False) for var in generator.trainable_variables]
discriminator_momentums = [tf.Variable(tf.zeros_like(var), trainable=False) for var in discriminator.trainable_variables]
discriminator_velocities = [tf.Variable(tf.zeros_like(var), trainable=False) for var in discriminator.trainable_variables]

@tf.function 
def train_step_phase4(real_images):
    current_batch_size = tf.shape(real_images)[0]
    
    # Keeping it at 100 so it matches your Epoch 40 Checkpoint!
    noise = tf.random.normal([current_batch_size, 1, 1, 100])

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        
        # --- THE FIX: Unpack using your original correct signature ---
        # We use underscores to ignore total_e, feats, and preds, and only capture the errs list
        generated_images, _, _, _, g_errs = generator(noise, training=True) 
        real_output, _, _, _, d_r_errs = discriminator(real_images, training=True)
        fake_output, _, _, _, d_f_errs = discriminator(generated_images, training=True)

        # Adversarial Losses
        g_adv_loss = cross_entropy(tf.ones_like(fake_output), fake_output)
        d_adv_loss = cross_entropy(tf.ones_like(real_output), real_output) + \
                     cross_entropy(tf.zeros_like(fake_output), fake_output)

        # --- THE FIX: SCALED PREDICTIVE CODING USING LIST INDEXING ---
        # Extract each error from the list explicitly to square and mean them
        raw_gen_pc = tf.reduce_mean(tf.square(g_errs[0])) + tf.reduce_mean(tf.square(g_errs[1])) + \
                     tf.reduce_mean(tf.square(g_errs[2])) + tf.reduce_mean(tf.square(g_errs[3]))
        scaled_gen_pc_loss = raw_gen_pc * LAMBDA_PC

        raw_disc_pc_real = tf.reduce_mean(tf.square(d_r_errs[0])) + tf.reduce_mean(tf.square(d_r_errs[1])) + \
                           tf.reduce_mean(tf.square(d_r_errs[2])) + tf.reduce_mean(tf.square(d_r_errs[3]))
        
        raw_disc_pc_fake = tf.reduce_mean(tf.square(d_f_errs[0])) + tf.reduce_mean(tf.square(d_f_errs[1])) + \
                           tf.reduce_mean(tf.square(d_f_errs[2])) + tf.reduce_mean(tf.square(d_f_errs[3]))
        
        scaled_disc_pc_loss = (raw_disc_pc_real + raw_disc_pc_fake) * LAMBDA_PC

        # Total Losses
        total_gen_loss = g_adv_loss + scaled_gen_pc_loss
        total_disc_loss = d_adv_loss + scaled_disc_pc_loss

    # 4. CALCULATE GRADIENTS
    gen_gradients = gen_tape.gradient(total_gen_loss, generator.trainable_variables)
    disc_gradients = disc_tape.gradient(total_disc_loss, discriminator.trainable_variables)
    
    # --- THE FIX: GRADIENT CLIPPING ---
    gen_gradients, _ = tf.clip_by_global_norm(gen_gradients, CLIP_NORM)
    disc_gradients, _ = tf.clip_by_global_norm(disc_gradients, CLIP_NORM)

    # 5. MANUAL ADAM-W UPDATE LOOP (GENERATOR)
    for i, (grad, var) in enumerate(zip(gen_gradients, generator.trainable_variables)):
        if grad is not None:
            m = generator_momentums[i]
            v = generator_velocities[i]
            
            m.assign(BETA_1 * m + (1.0 - BETA_1) * grad)
            v.assign(BETA_2 * v + (1.0 - BETA_2) * tf.square(grad))
            
            update = (MANUAL_LR * m / (tf.sqrt(v) + EPSILON)) + (WEIGHT_DECAY * MANUAL_LR * var)
            var.assign_sub(update)

    # 6. MANUAL ADAM-W UPDATE LOOP (DISCRIMINATOR)
    for i, (grad, var) in enumerate(zip(disc_gradients, discriminator.trainable_variables)):
        if grad is not None:
            m = discriminator_momentums[i]
            v = discriminator_velocities[i]
            
            m.assign(BETA_1 * m + (1.0 - BETA_1) * grad)
            v.assign(BETA_2 * v + (1.0 - BETA_2) * tf.square(grad))
            
            update = (MANUAL_LR * m / (tf.sqrt(v) + EPSILON)) + (WEIGHT_DECAY * MANUAL_LR * var)
            var.assign_sub(update)
            
    return total_gen_loss, total_disc_loss

In [11]:
TOTAL_PREVIOUS_EPOCHS = 40

In [ ]:
import time
import matplotlib.pyplot as plt
import tensorflow as tf

# 1. Set how long you want to train
EPOCHS = 10 

# 2. Create a static seed so we can see how the SAME images evolve
num_examples_to_generate = 16 # Generate a 4x4 grid
seed = tf.random.normal([num_examples_to_generate, 1, 1, 100])

def generate_and_save_images(model, epoch, test_input):
    # [CHANGED]: Catch all 5 returns from the new Phase 3 architecture
    predictions, _, _, _, _ = model(test_input, training=False)

    fig = plt.figure(figsize=(4, 4))

    for i in range(predictions.shape[0]):
        plt.subplot(4, 4, i+1)
        # Scale back from [-1, 1] to [0, 1] for matplotlib
        img = (predictions[i, :, :, :] + 1.0) / 2.0
        plt.imshow(img)
        plt.axis('off')

    plt.savefig(f'image_at_epoch_{epoch:04d}.png')
    # OPTIMIZATION 1: Explicitly close the 'fig' object. 
    plt.close(fig) 

def train(dataset, epochs):
    for epoch in range(epochs):
        start = time.time()
        
        # Container to record loss progress
        gen_loss_total = 0.0
        disc_loss_total = 0.0
        batch_count = 0

        actual_epoch = epoch + 1 + TOTAL_PREVIOUS_EPOCHS
        print(f"\nEpoch {actual_epoch} (Phase 4 New Model W/ Decentralized) is running...")
        
        for image_batch in dataset:
            g_loss, d_loss = train_step_phase4(image_batch)
            
            # OPTIMIZATION 2: Cast the tensor losses to pure Python floats.
            gen_loss_total += float(g_loss)
            disc_loss_total += float(d_loss)
            batch_count += 1
            
        # Calculate average to track performance
        avg_g_loss = gen_loss_total / batch_count
        avg_d_loss = disc_loss_total / batch_count
        
        print(f"Finished in {time.time()-start:.2f} seconds")
        print(f"Gen Loss: {avg_g_loss:.4f} | Disc Loss: {avg_d_loss:.4f}")
        
        # Save the model
        checkpoint_manager.save() 
        
        # Pass actual_epoch so it saves sequentially (e.g., image_at_epoch_0061.png)
        generate_and_save_images(generator, actual_epoch, seed)

# GO! Start Training
train(train_ds, EPOCHS)


Epoch 41 (Phase 4 New Model W/ Decentralized) is running...
Finished in 275.14 seconds
Gen Loss: 5.9885 | Disc Loss: 0.5902

Epoch 42 (Phase 4 New Model W/ Decentralized) is running...
Finished in 260.82 seconds
Gen Loss: 6.7895 | Disc Loss: 0.4019

Epoch 43 (Phase 4 New Model W/ Decentralized) is running...
Finished in 260.11 seconds
Gen Loss: 3.9206 | Disc Loss: 0.7918

Epoch 44 (Phase 4 New Model W/ Decentralized) is running...


In [13]:
from huggingface_hub import HfApi, create_repo

# 1. Initialize the API with your HF token
# Make sure hf_token is defined or retrieved from Kaggle Secrets
api = HfApi(token=hf_token)

# 2. Define your repository ID
repo_id = "RinKana/PC-GAN-V3-A"

# 3. Create the repository on Hugging Face (skips if already exists)
try:
    create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)
    print(f"Repository {repo_id} is ready!")
except Exception as e:
    print(f"Failed to create repository: {e}")

# 4. Upload the training checkpoints folder
# This ensures all epoch weights are preserved
print("Uploading checkpoint folder to Hugging Face...")
api.upload_folder(
    folder_path="./training_checkpoints", 
    repo_id=repo_id,
    repo_type="model",
    path_in_repo="checkpoints" 
)

# 5. Optional: Upload the notebook or source code
# Crucial for reproducing the PC-block architecture later
# api.upload_file(
#     path_or_fileobj="your_notebook.ipynb", 
#     path_in_repo="notebook.ipynb", 
#     repo_id=repo_id
# )

print(f"Upload successful! Model is secured at: https://huggingface.co/{repo_id}")

Repository RinKana/PC-GAN-V3-A is ready!
Uploading checkpoint folder to Hugging Face...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Upload successful! Model is secured at: https://huggingface.co/RinKana/PC-GAN-V3-A


In [ ]:
from huggingface_hub import HfApi, create_repo
import os

# 1. Initialize API (ensure hf_token is available in your environment)
api = HfApi(token=hf_token)

# 2. Define repository ID
repo_id = "RinKana/PC-GAN-V2-A"

# 3. Create repo if it doesn"t exist
try:
    create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)
    print(f"Repository {repo_id} is ready!")
except Exception as e:
    print(f"Repository status: {e}")

# 4. Upload images to "picture/epoch_1-40"
print("Starting batch upload for epoch images...")

# [OPTIMIZED]: Using upload_folder instead of a sequential loop for better performance.
# This uses concurrent uploads and reduces the number of commits.
api.upload_folder(
    folder_path="/kaggle/working",
    path_in_repo="picture/epoch_1-40",
    repo_id=repo_id,
    repo_type="model",
    allow_patterns="image_at_epoch_00*.png"
)

print(f"Success! All images uploaded to: https://huggingface.co/{repo_id}/tree/main/picture/epoch_1-40")